# Human disease genetics and drug targets along the PT (notebook 108)

**The question.** Where along the proximal tubule do inherited-tubulopathy genes, kidney-trait GWAS loci and renal drug
transporters sit, in human and mouse? The literature predicts positions for several; the continuous axis tests them.

**The frozen protocol** is `docs/results/pt-disease-genetics.md` (commit f5cfe25).
- H1: Mendelian proximal genes skew toward S1 (mean E below an expression-matched null).
- H2: urine-metabolite GWAS transporters are more graded than other transporters.
- H3: urate GWAS genes change pattern between species (reversed or species-only).
- H4: named positions (SGLT2 early; S3 module late; OAT1/3 mid-peak; OCT2 late in mouse only; LRP2/CUBN flat;
  GATM early; SLC4A4 early; URAT1 late in human, Glut9 not graded in mouse).
- H5: drug-target curves (descriptive).

**Sections.** 1 setup and gene sets; 2 H1–H3; 3 H4; 4 verdicts; 5 figures; 6 reading and run record.

## 1 · Setup and gene sets

In [ ]:
import argparse
import glob
import json
import os
import sys
import time
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
T71 = results_root / 'pt_pathways_package_pseudospace'
T90 = results_root / 'pt_continuous_axis_experiments' / 'tables'
T62 = results_root / 'pt_non_one_to_one_orthologs' / 'tables'
PANELS = data_root / 'external' / 'panelapp_2026-10-18'
GWAS = data_root / 'external' / 'gwas_catalog_v2_2026-10-18'
HGNC = data_root / 'external' / 'hgnc'
output = results_root / 'pt_disease_genetics'
tables, figures = output / 'tables', output / 'figures'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)

NOTEBOOK_LOGIC_VERSION = '108.disease_genetics.1'
SEED = 108
N_NULL, GRADED, PEAK_DROP = 2000, .5, .5
MOUSE, HUMAN = ['Ctrl1A2', 'Ctrl1A4'], ['HUK1_COR1', 'HUK1_MED1']
SPECIES = {'mouse': MOUSE, 'human': HUMAN}
PROX_WORDS = ['fanconi', 'dent', 'lowe', 'cystinosis', 'cystinuria', 'glucosuria', 'glycosuria', 'hartnup', 'iminoglycinuria',
              'proximal renal tubular acidosis', 'hypophosphat', 'rickets', 'aminoaciduria', 'lysinuric']
DIST_WORDS = ['bartter', 'gitelman', 'distal renal tubular acidosis', 'pseudohypoaldosteronism', 'nephrogenic diabetes insipidus',
              'hypomagnes', 'liddle', 'east']
ENDOCYTIC = ['LRP2', 'CUBN', 'AMN']
GWAS_PMIDS = {'eGFR': [34272381], 'URATE': [31578528], 'UMET': [31959995, 37277652]}
for path in [T90 / 'd_end_contrasts.csv.gz', T62 / 'gene_inventory_mouse.csv', T62 / 'gene_inventory_human.csv',
             T71 / 'tables' / 'universe_human_vs_mouse.csv', *[PANELS / f'panel_{i}.json' for i in (292, 149, 3993)],
             HGNC / 'hgnc_group_752.tsv', HGNC / 'hgnc_group_417.tsv']:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from pseudospace.external_replication import matched_means

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
SPECIES_COLOR = {'mouse': '#0072B2', 'human': '#009E73'}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_rows', 80)


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')


guard_log = []
END = pd.read_csv(T90 / 'd_end_contrasts.csv.gz')
END['human'] = END.human.astype(str)
for sp, pair in SPECIES.items():
    END[f'E {sp}'] = END[[f'E {s}' for s in pair]].mean(axis=1)
    END[f'early {sp}'] = (END[[f'E {s}' for s in pair]] <= -GRADED).all(axis=1)
    END[f'late {sp}'] = (END[[f'E {s}' for s in pair]] >= GRADED).all(axis=1)
UNI = END.set_index('human')
assert UNI.index.is_unique and len(UNI) == 7971, 'G1: universe'
inv = {'mouse': pd.read_csv(T62 / 'gene_inventory_mouse.csv').drop_duplicates('gene').set_index('gene'),
       'human': pd.read_csv(T62 / 'gene_inventory_human.csv').drop_duplicates('gene').set_index('gene')}
UNI['level mouse'] = inv['mouse'].log2_cp10k.reindex(UNI.mouse).to_numpy()
UNI['level human'] = inv['human'].log2_cp10k.reindex(UNI.index).to_numpy()
for sp in SPECIES:
    UNI[f'decile {sp}'] = pd.qcut(UNI[f'level {sp}'].rank(method='first'), 10, labels=False).fillna(-1).astype(int)

# gene sets
def phenotype_text(gene):
    return ' '.join(map(str, gene.get('phenotypes') or [])).lower()


prox, dist = set(), set()
for pid in (292, 149, 3993):
    panel = json.loads((PANELS / f'panel_{pid}.json').read_text())
    for g in panel['genes']:
        if str(g.get('confidence_level')) != '3':
            continue
        sym = g['entity_name']
        text = phenotype_text(g)
        is_prox = any(w in text for w in PROX_WORDS)
        if is_prox:
            prox.add(sym)
        if pid == 292 and not is_prox and any(re.search(r'\b' + re.escape(w) + r'\b', text) for w in DIST_WORDS):
            dist.add(sym)
prox |= set(ENDOCYTIC)
dist -= prox


def gwas_genes(pmids):
    out = set()
    for pmid in pmids:
        for f in sorted(GWAS.glob(f'assoc_{pmid}_p*.json')):
            for a in json.loads(f.read_text()).get('_embedded', {}).get('associations', []):
                for mg in a.get('mapped_genes') or []:
                    out |= {x.strip() for x in re.split(r'[,;]| - ', str(mg)) if x.strip()}
    return out


TRANSPORTERS = set(pd.concat([pd.read_csv(HGNC / f, sep='\t') for f in ('hgnc_group_752.tsv', 'hgnc_group_417.tsv')])['Approved symbol'].astype(str))
SETS = {'PROX': prox, 'DIST': dist, **{k: gwas_genes(v) for k, v in GWAS_PMIDS.items()}}
SETS.update({f'{k}-TR': SETS[k] & TRANSPORTERS for k in ('eGFR', 'URATE', 'UMET')})
SETS['TR'] = TRANSPORTERS
rows = []
for name, genes in SETS.items():
    for g in sorted(genes):
        inu = g in UNI.index
        rows.append({'set': name, 'gene': g, 'in universe': inu,
                     **({k: UNI.loc[g, k] for k in ('mouse', 'E mouse', 'E human', 'category', 'early mouse', 'late mouse',
                                                    'early human', 'late human')} if inu else {})})
SET_TABLE = pd.DataFrame(rows)
SET_TABLE.to_csv(tables / 'sets.csv', index=False)
SIZES = SET_TABLE.groupby('set')['in universe'].agg(['size', 'sum']).rename(columns={'size': 'genes', 'sum': 'in universe'})
display(SIZES)
print('PROX in universe:', ', '.join(sorted(g for g in prox if g in UNI.index)))
print('DIST in universe:', ', '.join(sorted(g for g in dist if g in UNI.index)))
guard_log.append('G2: set sizes (all / in the 1:1 universe): ' + '; '.join(f'{k} {r.genes}/{r["in universe"]}' for k, r in SIZES.iterrows()))

## 2 · H1–H3: sets against expression-matched nulls

In [ ]:
def null_test(genes, values, sp, *, pool=None, seed=SEED):
    """Observed mean of ``values`` over ``genes`` against matched random lists from ``pool`` (default: universe)."""
    pool_idx = UNI.index if pool is None else pool
    genes = [g for g in genes if g in pool_idx and np.isfinite(values.get(g, np.nan))]
    pool_idx = [g for g in pool_idx if np.isfinite(values.get(g, np.nan))]
    strata = UNI.loc[pool_idx, f'decile {sp}'].to_numpy()
    list_strata = UNI.loc[genes, f'decile {sp}'].to_numpy()
    null = matched_means(values.reindex(pool_idx).to_numpy()[:, None], strata, list_strata, n=N_NULL, seed=seed)[:, 0]
    obs = float(values.reindex(genes).mean())
    return {'genes': len(genes), 'observed': obs, 'null mean': float(null.mean()), 'null 5%': float(np.quantile(null, .05)),
            'null 95%': float(np.quantile(null, .95)), 'p low': float((1 + (null <= obs).sum()) / (N_NULL + 1)),
            'p high': float((1 + (null >= obs).sum()) / (N_NULL + 1))}


res = []
for sp in SPECIES:
    E = UNI[f'E {sp}']
    graded = (UNI[f'early {sp}'] | UNI[f'late {sp}']).astype(float)
    r = null_test(SETS['PROX'], E, sp, seed=SEED)
    res.append({'test': 'H1 PROX mean E (below null = toward S1)', 'species': sp, **r, 'holds': r['observed'] < r['null 5%']})
    r = null_test(SETS['DIST'], graded, sp, seed=SEED + 1)
    res.append({'test': 'H1 DIST share graded (comparator)', 'species': sp, **r, 'holds': None})
    r = null_test(SETS['PROX'], graded, sp, seed=SEED + 2)
    res.append({'test': 'H1 PROX share graded (descriptive)', 'species': sp, **r, 'holds': None})
    tr_pool = pd.Index([g for g in SETS['TR'] if g in UNI.index])
    for k in ('UMET-TR', 'eGFR-TR'):
        r = null_test(SETS[k], E.abs(), sp, pool=tr_pool, seed=SEED + 3)
        res.append({'test': f'H2 {k} mean |E| among transporters', 'species': sp, **r,
                    'holds': (r['observed'] > r['null 95%']) if k == 'UMET-TR' else None})
changed = UNI.category.isin(['reversed', 'mouse only', 'human only']).astype(float)
r = null_test(SETS['URATE'], changed, 'human', seed=SEED + 4)
res.append({'test': 'H3 URATE share reversed or species-only', 'species': 'both', **r, 'holds': r['observed'] > r['null 95%']})
r = null_test(SETS['URATE-TR'], changed, 'human', seed=SEED + 5)
res.append({'test': 'H3 URATE-TR share reversed or species-only (descriptive)', 'species': 'both', **r, 'holds': None})
H13 = pd.DataFrame(res)
H13.to_csv(tables / 'h1_h3.csv', index=False)
display(H13.round(3))

## 3 · H4: named positions

In [ ]:
def stored(folder, stage):
    files = sorted((folder / 'stage_cache').glob(f'{stage}__*.npz'),
                   key=lambda p: json.loads(Path(str(p) + '.key.json').read_text())['written_utc'])
    if not files:
        raise FileNotFoundError(f'{folder.name}: no cached {stage}')
    with np.load(files[-1], allow_pickle=False) as payload:
        return {k: payload[k] for k in payload.files}


U71 = pd.read_csv(T71 / 'tables' / 'universe_human_vs_mouse.csv').gene.astype(str).tolist()
payload = stored(T71, 'gene_model_human_vs_mouse_species')
GRID = payload['grid'].astype(float)
CURVES = {n: pd.DataFrame(payload[f'curve|{n}'], index=U71) / np.log(2) for n in MOUSE + HUMAN}   # log2 rate
guard_log.append(f"G3: notebook 71's species curves for {len(U71):,} genes on {len(GRID)} grid points")
M2H = dict(zip(UNI.mouse, UNI.index))
H2M = dict(zip(UNI.index, UNI.mouse))


def species_curve(human_symbol, sp):
    m = H2M.get(human_symbol)
    if m is None or m not in CURVES[MOUSE[0]].index:
        return None
    return np.nanmean([CURVES[s].loc[m].to_numpy() for s in SPECIES[sp]], axis=0)


def mid_peak(human_symbol, sp):
    c = species_curve(human_symbol, sp)
    if c is None:
        return None
    ok = np.isfinite(c)
    g, c = GRID[ok], c[ok]
    top = int(np.argmax(c))
    tail = c[g >= g[0] + .9 * (g[-1] - g[0])].mean()
    return bool(.2 <= g[top] <= .8 and c[top] - tail >= PEAK_DROP)


def state(g, sp):
    if g not in UNI.index:
        return 'not in universe'
    return 'early' if UNI.loc[g, f'early {sp}'] else 'late' if UNI.loc[g, f'late {sp}'] else 'not graded'


checks = []
def add(label, gene, sp, observed, met):
    checks.append({'check': label, 'gene': gene, 'species': sp, 'observed': observed,
                   'E': UNI.loc[gene, f'E {sp}'] if gene in UNI.index else np.nan, 'met': met})


for sp in SPECIES:
    add('a SLC5A2 early', 'SLC5A2', sp, state('SLC5A2', sp), state('SLC5A2', sp) == 'early')
    for g in ('SLC7A13', 'SLC22A7', 'SLC3A1'):
        add('b S3 module late', g, sp, state(g, sp), state(g, sp) == 'late' if g in UNI.index else None)
    for g in ('SLC22A6', 'SLC22A8'):
        mp = mid_peak(g, sp)
        add('c OAT1/OAT3 peak inside the axis', g, sp, 'mid-peak' if mp else ('no curve' if mp is None else 'no mid-peak'), mp)
    for g in ('LRP2', 'CUBN'):
        add('e endocytic receptors not graded', g, sp, state(g, sp), state(g, sp) == 'not graded' if g in UNI.index else None)
    add('f GATM early', 'GATM', sp, state('GATM', sp), state('GATM', sp) == 'early' if 'GATM' in UNI.index else None)
    add('f EHHADH not early', 'EHHADH', sp, state('EHHADH', sp), state('EHHADH', sp) != 'early' if 'EHHADH' in UNI.index else None)
    add('g SLC4A4 early', 'SLC4A4', sp, state('SLC4A4', sp), state('SLC4A4', sp) == 'early' if 'SLC4A4' in UNI.index else None)
add('b SLC5A11 late (human)', 'SLC5A11', 'human', state('SLC5A11', 'human'), state('SLC5A11', 'human') == 'late' if 'SLC5A11' in UNI.index else None)
add('d OCT2 late in mouse', 'SLC22A2', 'mouse', state('SLC22A2', 'mouse'), state('SLC22A2', 'mouse') == 'late' if 'SLC22A2' in UNI.index else None)
add('d OCT2 not late in human', 'SLC22A2', 'human', state('SLC22A2', 'human'), state('SLC22A2', 'human') != 'late' if 'SLC22A2' in UNI.index else None)
add('h URAT1 late in human', 'SLC22A12', 'human', state('SLC22A12', 'human'), state('SLC22A12', 'human') == 'late' if 'SLC22A12' in UNI.index else None)
add('h Glut9 not graded in mouse', 'SLC2A9', 'mouse', state('SLC2A9', 'mouse'), state('SLC2A9', 'mouse') == 'not graded' if 'SLC2A9' in UNI.index else None)
H4 = pd.DataFrame(checks)
H4.to_csv(tables / 'h4_named.csv', index=False)
display(H4.round(3))
print(f"H4: {int(H4.met.eq(True).sum())} met, {int(H4.met.eq(False).sum())} missed, {int(H4.met.isna().sum())} not testable")

### 3.1 · Post hoc: are the graded distal-disease genes neighbouring-structure RNA?

Added after H1 showed DIST genes graded more often than the null. Notebook 104's edge enrichment (ε, human vs mouse
run) and core-rebuild verdict are joined for the DIST genes and the H4 misses.

In [ ]:
T104 = results_root / 'pt_contamination_audit' / 'tables'
eps104 = pd.read_csv(T104 / 'edge_enrichment.csv.gz', index_col=0)
core104 = pd.read_csv(T104 / 'gradient_robustness.csv.gz')
look = sorted({g for g in SETS['DIST'] if g in UNI.index} | {'SLC2A9', 'SLC3A1'})
POSTHOC = pd.DataFrame({'gene': look, 'mouse': [H2M[g] for g in look],
                        'state mouse': [state(g, 'mouse') for g in look], 'state human': [state(g, 'human') for g in look]})
for col in [c for c in eps104.columns if c.startswith('hvm')]:
    POSTHOC[f'ε {col.split()[1]}'] = eps104[col].reindex(POSTHOC.mouse).to_numpy()
for sp in SPECIES:
    sub = core104[core104.species.eq(sp)].set_index('gene')
    POSTHOC[f'survives core {sp}'] = sub['survives core3'].reindex(POSTHOC.mouse).to_numpy()
POSTHOC.to_csv(tables / 'posthoc_distal_genes.csv', index=False)
display(POSTHOC.round(2))

## 4 · Verdicts

In [ ]:
def fmt(row):
    return f"{row.observed:.3f} (null {row['null 5%']:.3f}–{row['null 95%']:.3f}; n {row.genes})"


VERDICTS = pd.DataFrame([
    *[(f'{r.test} · {r.species}', f'{fmt(r)}: {r.holds}') for _, r in H13.iterrows()],
    ('H4 named positions', f"{int(H4.met.eq(True).sum())} met / {int(H4.met.eq(False).sum())} missed / {int(H4.met.isna().sum())} not testable"),
], columns=['reading', 'result'])
VERDICTS.to_csv(tables / 'verdicts.csv', index=False)
display(VERDICTS)

## 5 · Figures (plots only)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(120 * MM, 45 * MM), sharey=True)
for ax, sp in zip(axes, SPECIES):
    sub = H13[H13.species.eq(sp) & H13.test.str.contains('mean')]
    y = np.arange(len(sub))
    ax.hlines(y, sub['null 5%'], sub['null 95%'], color=LIGHT, lw=3)
    ax.scatter(sub.observed, y, color=SPECIES_COLOR[sp], s=12, zorder=3)
    ax.set_yticks(y, [t.split(' ', 1)[1] for t in sub.test])
    ax.set_title(sp)
save(fig, '108a_sets_against_null')
plt.show()

named = ['SLC5A2', 'SLC7A13', 'SLC22A7', 'SLC3A1', 'SLC22A6', 'SLC22A8', 'SLC22A2', 'LRP2', 'CUBN', 'GATM', 'SLC4A4', 'SLC22A12']
named = [g for g in named if g in UNI.index]
fig, axes = plt.subplots(2, 6, figsize=(180 * MM, 60 * MM), sharex=True)
for ax, g in zip(axes.ravel(), named):
    for sp in SPECIES:
        c = species_curve(g, sp)
        if c is not None:
            ax.plot(GRID, c - np.nanmean(c), color=SPECIES_COLOR[sp])
    ax.set_title(g, fontsize=6)
for ax in axes[-1]:
    ax.set_xlabel('PT position')
for ax in axes.ravel()[len(named):]:
    ax.axis('off')
save(fig, '108b_named_curves')
plt.show()

drugs = [g for g in ('SLC5A2', 'SLC22A12', 'SLC22A6', 'SLC22A2', 'LRP2') if g in UNI.index]
fig, axes = plt.subplots(1, len(drugs), figsize=(30 * MM * len(drugs), 35 * MM), sharex=True)
for ax, g in zip(np.atleast_1d(axes), drugs):
    for sp in SPECIES:
        c = species_curve(g, sp)
        if c is not None:
            ax.plot(GRID, c - np.nanmean(c), color=SPECIES_COLOR[sp])
    ax.set_title(g, fontsize=6); ax.set_xlabel('PT position')
save(fig, '108c_drug_targets')
plt.show()

## 6 · How the protocol was read, and the run record

In [ ]:
READING = pd.DataFrame([
    ('reading', 'Panel genes are matched on the PanelApp entity_name (HGNC symbol) to notebook 90\'s human symbols; the PROX keywords are searched in each green gene\'s phenotype strings, the DIST keywords as whole words.'),
    ('reading', 'GWAS mapped_genes strings are split on commas, semicolons and " - "; all mapped genes of an association count.'),
    ('reading', 'Expression deciles are over the 1:1 universe from notebook 62\'s log2 CP10k of that species; H3 uses human deciles.'),
    ('reading', 'Curves are notebook 71\'s species specimen curves converted to log2 and averaged over the species\' two specimens; the H4c tail is the last 10% of the grid span.'),
], columns=['kind', 'what'])
READING.to_csv(tables / 'how_the_protocol_was_read.csv', index=False)
RUN = {'notebook': '108_pt_disease_genetics', 'logic_version': NOTEBOOK_LOGIC_VERSION, 'seed': SEED, 'guards': guard_log,
       'sets': SIZES.reset_index().to_dict('records'), 'h1_h3': H13.round(4).to_dict('records'),
       'h4': H4.round(3).to_dict('records'), 'seconds': round(time.perf_counter() - NOTEBOOK_START, 1)}
(output / 'run_record.json').write_text(json.dumps(RUN, indent=2, default=str))
print(json.dumps({k: RUN[k] for k in ('sets', 'seconds')}, indent=2, default=str))